Below is a **simple production-style RAG example** using the **Azure ecosystem**. This is exactly the level of code you can explain in an interview.

---

# Architecture

```text
PDF Documents
      │
      ▼
PyPDFLoader
      │
      ▼
RecursiveCharacterTextSplitter
      │
      ▼
Azure OpenAI Embeddings
      │
      ▼
Azure AI Search (Vector Index)
      │
      ▼
Retriever
      │
      ▼
Prompt Template
      │
      ▼
Azure OpenAI (GPT-4o)
      │
      ▼
Final Answer
```

---

# Install Packages

```bash
pip install langchain
pip install langchain-openai
pip install langchain-community
pip install langchain-text-splitters
pip install langchain-azure-ai
pip install pypdf
```

---

# Step 1: Import Libraries

```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
```

---

# Step 2: Configure Azure OpenAI

```python
import os

os.environ["AZURE_OPENAI_API_KEY"] = "YOUR_API_KEY"

os.environ["AZURE_OPENAI_ENDPOINT"] = \
"https://my-openai.openai.azure.com"

os.environ["OPENAI_API_VERSION"] = "2024-02-15-preview"
```

---

# Step 3: Load Documents

```python
loader = PyPDFLoader("policy.pdf")

documents = loader.load()
```

---

# Step 4: Split Documents

```python
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = splitter.split_documents(documents)
```

---

# Step 5: Azure OpenAI Embeddings

```python
embeddings = AzureOpenAIEmbeddings(

    model="text-embedding-3-small",

    azure_deployment="embedding-deployment"
)
```

---

# Step 6: Store in Azure AI Search

```python
from langchain_community.vectorstores import AzureSearch

vector_store = AzureSearch.from_documents(

    documents=docs,

    embedding=embeddings,

    azure_search_endpoint="https://mysearch.search.windows.net",

    azure_search_key="SEARCH_ADMIN_KEY",

    index_name="policy-index"
)
```

---

# Step 7: Create Retriever

```python
retriever = vector_store.as_retriever(

    search_kwargs={"k":3}
)
```

---

# Step 8: Azure OpenAI LLM

```python
llm = AzureChatOpenAI(

    azure_deployment="gpt-4o",

    api_version="2024-02-15-preview",

    temperature=0
)
```

---

# Step 9: Prompt

```python
prompt = ChatPromptTemplate.from_template("""

Answer only from the given context.

Context:

{context}

Question:

{question}

If the answer is unavailable, say:

"I don't know."

""")
```

---

# Step 10: Build LCEL Chain

```python
chain = (

    {

        "context": retriever,

        "question": RunnablePassthrough()

    }

    | prompt

    | llm

    | StrOutputParser()

)
```

---

# Step 11: Ask Question

```python
response = chain.invoke(

    "What is the leave policy?"

)

print(response)
```

---

# Interview Explanation (1 Minute)

> First, we load the PDF using **PyPDFLoader** and split it into chunks with **RecursiveCharacterTextSplitter**. Each chunk is converted into embeddings using **Azure OpenAI Embeddings** (`text-embedding-3-small`). These vectors are stored in **Azure AI Search**, which acts as the vector database. When a user asks a question, the retriever performs a similarity search and returns the top 3 relevant chunks. LangChain injects the retrieved context into the prompt and sends it to **Azure OpenAI GPT-4o**. The model generates the answer using only the retrieved context, reducing hallucinations.

---

# Mapping to Your AWS Bedrock Project

| AWS Bedrock Project | Azure Version |
|---------------------|---------------|
| Titan Embeddings | Azure OpenAI Embeddings |
| Qdrant | Azure AI Search |
| Claude | GPT-4o |
| Bedrock Runtime | Azure OpenAI |
| IAM Role | Managed Identity |
| CloudWatch | Azure Monitor |

---

# If They Ask: "Can you migrate your Bedrock RAG to Azure?"

You can confidently answer:

> "Yes. The application architecture remains the same. I would replace the Bedrock embedding model with Azure OpenAI Embeddings, replace Qdrant with Azure AI Search (or keep Qdrant if required), replace Claude with an Azure OpenAI deployment such as GPT-4o, and update authentication to use Managed Identity instead of IAM roles. The LangChain orchestration and RAG flow remain largely unchanged."

This answer demonstrates that you understand both the cloud-specific services and the cloud-agnostic nature of LangChain-based RAG applications.